# EHRGym GRPO Training with TRL + OpenEnv

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adtserapio/EHRGym/blob/main/notebooks/ehrgym_grpo_training.ipynb)
[![HF Space](https://img.shields.io/badge/%F0%9F%8F%A5-EHRGym%20Space-blue)](https://huggingface.co/spaces/openenv-community/EHRGym)

Train a language model to operate an Epic-style Electronic Health Records (EHR) system using **GRPO** (Group Relative Policy Optimization) via [TRL](https://github.com/huggingface/trl) and the [OpenEnv](https://huggingface.co/docs/trl/openenv) framework.

The agent learns to:
- Navigate an EHR interface (patient charts, labs, notes, orders)
- Place clinical orders (IV fluids, labs, medications)
- Write SOAP-style clinical notes
- Sign encounters to complete clinical workflows

**Architecture:**
```
┌────────────────────────────────────────┐
│  GRPOTrainer (GPU)                     │
│  ┌────────┐  ┌──────────┐  ┌────────┐ │
│  │ Model  │→ │ rollout  │→ │EHRGym  │ │
│  │(Qwen3) │← │ func    │← │Env     │ │
│  └────────┘  └──────────┘  └───┬────┘ │
└────────────────────────────────┼───────┘
                                │ HTTP
┌────────────────────────────────┼───────┐
│  EHRGym Server (Docker/Space)  ▼       │
│  FastAPI → Playwright → Next.js EHR    │
└────────────────────────────────────────┘
```

## Install dependencies

We install **TRL** with vLLM support, and **EHRGym** directly from GitHub.

In [ ]:
!pip install -Uq "trl[vllm]" git+https://github.com/adtserapio/EHRGym.git trackio

### Log in to Hugging Face

Log in to save your fine-tuned model and track experiments on the Hub.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Initialize the EHRGym Environment

EHRGym exposes a standard OpenEnv API (`reset()` / `step()`) backed by a real browser (Playwright + Chromium) interacting with a Next.js EHR application.

You can connect to:
- The **hosted Space** on Hugging Face (easiest, no setup)
- A **local Docker container** (recommended for training throughput)

The environment provides 25 clinical tasks across three difficulty levels (Basic, Medium, Hard), spanning conditions like AKI, DKA, pneumonia, CHF, stroke, and more.

In [ ]:
from ehrgym import EHRGymEnv
from ehrgym.env import TASK_IDS

# Option A: Connect to hosted HF Space (no setup needed)
ehrgym_url = "https://openenv-community-ehrgym.hf.space"
env = EHRGymEnv(base_url=ehrgym_url)

# Option B: Connect to local Docker container
# docker run -d -p 7860:7860 registry.hf.space/openenv-community-ehrgym:latest
# env = EHRGymEnv(base_url="http://localhost:7860")

# Option C: Connect to local dev server
# env = EHRGymEnv(base_url="http://localhost:8000")

# Verify connectivity
state = env.get_state()
print(f"Connected to EHRGym | Status: {state.get('status', 'ok')}")
print(f"Available tasks ({len(TASK_IDS)}): {', '.join(TASK_IDS[:5])}...")

### Quick sanity check

Let's run a quick `reset()` and one `step()` to verify the environment works end-to-end.

In [ ]:
# Reset with a specific task
obs = env.reset(task_id="aki-chart-review")
print("=== Reset Observation ===")
print(obs)

# Take one action
result = env.navigate("/patient/pat-1001")
print("\n=== Step Result ===")
print(result)
print(f"\nReward: {env.cumulative_reward}, Steps: {env.step_count}, Done: {env.done}")

## Init model and tokenizer

We use [Qwen/Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B), a lightweight model suitable for agent training.  
For better clinical task performance, scale up to larger models (e.g., Qwen3.5-7B or Qwen3.5-32B).

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3.5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

## Define the system prompt

The system prompt establishes the clinical persona and available actions. The model acts as an attending physician using the EHR.

In [ ]:
system_prompt = """
You are a clinical AI agent operating an Epic-style Electronic Health Records (EHR) system.
You are logged in as Patrick Sullivan, MD — the attending physician.

## AVAILABLE ACTIONS

You interact with the EHR by outputting one action per turn in this exact format:

ACTION: <action_type> | <params>

Available actions:
- ACTION: navigate | <url>              — Go to a URL (e.g., /patient/pat-1001)
- ACTION: click | <css_selector>        — Click an element (e.g., button[data-testid='sign-encounter'])
- ACTION: type | <css_selector> | <text> — Type into an input field
- ACTION: press | <key>                 — Press a key (e.g., Enter, Tab)

## WORKFLOW

For each clinical task, you should:
1. Navigate to the patient's chart
2. Review relevant information (labs, notes, encounters)
3. Write your clinical assessment in the SOAP note
4. Place appropriate orders (medications, labs, imaging)
5. Sign orders and the encounter

## RESPONSE FORMAT

Respond with exactly one action per turn. Format:
ACTION: <type> | <params>

Example:
ACTION: navigate | /patient/pat-1001
ACTION: click | [data-testid='chart-review-tab']
ACTION: type | #soap-assessment | Patient presents with acute kidney injury...
ACTION: click | button:has-text('Sign Encounter')
"""

## Rollout function

The rollout function orchestrates multi-turn interaction between the model and the EHR environment.  
At each turn:

1. The model generates an action based on the current observation
2. The action is parsed and executed in the environment
3. Environment feedback (URL, rubric progress) is appended to the conversation
4. Rewards are collected for GRPO optimization

The `env_mask` ensures only **model-generated tokens** contribute to the training loss, while environment feedback tokens are masked out.

In [ ]:
import re


def parse_action(text: str) -> dict | None:
    """Parse the model's output into an EHRGym action dict.

    Expected format: ACTION: <type> | <params>
    """
    match = re.search(r"ACTION:\s*(navigate|click|type|press)\s*\|\s*(.+)", text, re.IGNORECASE)
    if not match:
        return None

    action_type = match.group(1).lower().strip()
    params = match.group(2).strip()

    if action_type == "navigate":
        return {"type": "goto", "url": params}
    elif action_type == "click":
        return {"type": "click", "selector": params}
    elif action_type == "type":
        parts = params.split("|", 1)
        if len(parts) == 2:
            return {"type": "fill", "selector": parts[0].strip(), "text": parts[1].strip()}
        return None
    elif action_type == "press":
        return {"type": "keypress", "key": params}
    return None


def format_action_result(env: EHRGymEnv, result_text: str) -> str:
    """Format environment feedback for the conversation history."""
    parts = [result_text]
    parts.append(f"Cumulative reward: {env.cumulative_reward:.2f}")
    if env.done:
        parts.append("STATUS: All rubric items complete.")
    return "\n".join(parts)

In [ ]:
from trl.experimental.openenv import generate_rollout_completions

max_new_tokens = 128  # EHR actions need more tokens than simple games
max_turns = 25       # Max browser actions per episode


def rollout_once(trainer, env, tokenizer, task_id, dataset_prompt, system_prompt, max_turns, max_new_tokens):
    """Run one full EHR episode: reset → navigate → chart review → orders → sign."""
    # Reset environment with the clinical task
    obs_text = env.reset(task_id=task_id)

    prompt_ids = []
    completion_ids = []
    logprobs = []
    env_mask = []  # 1 = model tokens, 0 = env tokens (masked from loss)
    model_outputs = []
    step_rewards = []

    # Build initial prompt
    accumulated_messages = [{"role": "system", "content": system_prompt}]
    initial_user_msg = f"{dataset_prompt}\n\nEnvironment: {obs_text}"
    initial_messages = accumulated_messages + [{"role": "user", "content": initial_user_msg}]

    initial_prompt_text = tokenizer.apply_chat_template(
        initial_messages,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False,
    )
    initial_prompt_ids = tokenizer.encode(initial_prompt_text, add_special_tokens=False)
    prompt_ids.extend(initial_prompt_ids)

    prev_env_len = 0

    for turn in range(max_turns):
        if env.done:
            break

        # Build prompt with conversation history
        user_msg = initial_user_msg if turn == 0 else accumulated_messages[-1]["content"]
        messages = accumulated_messages + [{"role": "user", "content": user_msg}] if turn > 0 else initial_messages
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        # Generate completion
        rollout_outputs = generate_rollout_completions(
            trainer, [prompt_text],
            generation_overrides={"max_tokens": max_new_tokens}
        )[0]

        # Add model tokens with newline separators
        newline_tokens = tokenizer.encode("\n", add_special_tokens=False)
        completion_ids.extend(newline_tokens)
        logprobs.extend([0.0] * len(newline_tokens))
        env_mask.extend([1] * len(newline_tokens))

        completion_ids.extend(rollout_outputs["completion_ids"])
        logprobs.extend(rollout_outputs["logprobs"])
        env_mask.extend([1] * len(rollout_outputs["completion_ids"]))

        completion_text = rollout_outputs.get("text") or tokenizer.decode(
            rollout_outputs["completion_ids"], skip_special_tokens=True
        )
        model_outputs.append(completion_text.strip())

        # Parse and execute action
        action = parse_action(completion_text)
        if action is None:
            # Bad format — give feedback but continue
            env_feedback = "ERROR: Could not parse action. Use format: ACTION: <type> | <params>"
            step_rewards.append(-0.1)
        else:
            # Execute action in the environment
            action_type = action["type"]
            if action_type == "goto":
                env_feedback = env.navigate(action["url"])
            elif action_type == "click":
                env_feedback = env.click(action["selector"])
            elif action_type == "fill":
                env_feedback = env.type_text(action["selector"], action["text"])
            elif action_type == "keypress":
                env_feedback = env.press_key(action["key"])
            else:
                env_feedback = "ERROR: Unknown action type."
            step_rewards.append(env._last_reward)

        env_feedback = format_action_result(env, env_feedback)

        # Add environment feedback tokens (masked from loss)
        env_tokens = tokenizer.encode("\n" + env_feedback, add_special_tokens=False)
        completion_ids.extend(env_tokens)
        logprobs.extend([0.0] * len(env_tokens))
        env_mask.extend([0] * len(env_tokens))  # Mask env tokens from loss

        # Update conversation history
        accumulated_messages.append({"role": "user", "content": user_msg if turn == 0 else env_feedback})
        accumulated_messages.append({"role": "assistant", "content": completion_text})
        if turn > 0:
            accumulated_messages.append({"role": "user", "content": env_feedback})

    # Compute final rewards
    task_reward_val = env.cumulative_reward
    done = env.done
    steps = env.step_count

    # Efficiency bonus: full bonus at <=10 steps, linear decay to 0 at 30
    if done and steps > 0:
        efficiency = max(0.0, 1.0 - max(0, steps - 10) / 20.0)
    else:
        efficiency = 0.0

    # Format reward: proportion of well-formatted actions
    if model_outputs:
        format_score = sum(1 for o in model_outputs if parse_action(o) is not None) / len(model_outputs)
    else:
        format_score = 0.0

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "env_mask": env_mask,
        "task_reward": task_reward_val,
        "efficiency_reward": efficiency,
        "format_reward": format_score,
        "model_outputs": model_outputs,
    }

In [ ]:
import random


def rollout_func(prompts, trainer):
    """Rollout function for GRPOTrainer.

    For each prompt, runs a full EHR episode and collects rewards.
    """
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    episode_env_masks = []
    task_rewards = []
    efficiency_rewards = []
    format_rewards = []

    for prompt_text in prompts:
        # Pick a random task for each episode
        task_id = random.choice(TASK_IDS)

        episode = rollout_once(
            trainer=trainer,
            env=env,
            tokenizer=tokenizer,
            task_id=task_id,
            dataset_prompt=prompt_text,
            system_prompt=system_prompt,
            max_turns=max_turns,
            max_new_tokens=max_new_tokens,
        )
        episode_prompt_ids.append(episode["prompt_ids"])
        episode_completion_ids.append(episode["completion_ids"])
        episode_logprobs.append(episode["logprobs"])
        episode_env_masks.append(episode["env_mask"])
        task_rewards.append(episode["task_reward"])
        efficiency_rewards.append(episode["efficiency_reward"])
        format_rewards.append(episode["format_reward"])

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "env_mask": episode_env_masks,
        "task_reward": task_rewards,
        "efficiency_reward": efficiency_rewards,
        "format_reward": format_rewards,
    }

## Define reward functions

Three reward signals guide the agent's learning:

- **`reward_task`** — Primary signal from the environment's clinical rubric (orders placed, note elements written, encounter signed)
- **`reward_format`** — Rewards correctly formatted actions; penalizes unparseable outputs
- **`reward_efficiency`** — Bonus for completing tasks in fewer steps

In [ ]:
def reward_task(completions, **kwargs):
    """Primary clinical task reward from environment rubric progress."""
    rewards = kwargs.get("task_reward")
    if rewards is None:
        return [0.0] * len(completions)
    return [float(r) for r in rewards]


def reward_format(completions, **kwargs):
    """Reward for correctly formatted ACTION outputs."""
    rewards = kwargs.get("format_reward")
    if rewards is None:
        return [0.0] * len(completions)
    return [float(r) for r in rewards]


def reward_efficiency(completions, **kwargs):
    """Bonus for completing tasks efficiently (fewer browser actions)."""
    rewards = kwargs.get("efficiency_reward")
    if rewards is None:
        return [0.0] * len(completions)
    return [float(r) for r in rewards]

## Create dataset

Each dataset entry triggers one rollout episode during training.  
The prompt provides the base instruction; the specific clinical task is sampled randomly at rollout time.

In [ ]:
from datasets import Dataset

dataset_size = 500  # Number of training episodes (increase for longer runs)
dataset_prompt = (
    "Complete the clinical chart review task. Review the patient's chart, "
    "write your clinical assessment, place appropriate orders, and sign the encounter."
)

dataset = Dataset.from_dict({"prompt": [dataset_prompt] * dataset_size})
print(f"Dataset: {len(dataset)} episodes")

## Set GRPO Config

Key parameters for EHRGym training:
- `max_completion_length=4096` — EHR episodes involve many turns with verbose observations
- `num_generations=2` — Each prompt gets 2 rollouts for variance reduction
- `vllm_gpu_memory_utilization=0.3` — Reserve GPU memory for vLLM inference

In [ ]:
from trl import GRPOConfig

output_dir = "ehrgym-grpo-qwen3.5-2b"

grpo_config = GRPOConfig(
    # Training schedule
    num_train_epochs=1,
    learning_rate=1e-6,
    gradient_accumulation_steps=16,
    per_device_train_batch_size=1,
    warmup_steps=10,
    optim="adamw_torch",
    max_grad_norm=1.0,

    # GRPO configuration
    num_generations=2,
    max_completion_length=4096,
    log_completions=False,

    # vLLM configuration
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.3,
    vllm_max_model_length=6144,

    # Logging
    output_dir=output_dir,
    report_to="trackio",
    logging_steps=1,
    save_steps=25,
    save_total_limit=2,

    # Memory optimization
    gradient_checkpointing=True,

    # Hub integration
    push_to_hub=True,
)

## Create GRPOTrainer and start training

The trainer manages the full RL loop: generating actions via vLLM, stepping through the EHR environment, computing rewards, and updating the policy.

In [ ]:
import sys
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[
        reward_task,
        reward_format,
        reward_efficiency,
    ],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")
else:
    print("No GPU detected. Training will be slow.")
    start_gpu_memory = 0
    max_memory = 0

In [ ]:
trainer_stats = trainer.train()

## Save and push model

In [ ]:
env.close()
trainer.save_model(output_dir)
trainer.push_to_hub()

## Training stats

In [ ]:
import torch

if torch.cuda.is_available():
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_training = round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory / max_memory * 100, 3)
    training_memory_percentage = round(used_memory_for_training / max_memory * 100, 3)

    print(f"{trainer_stats.metrics['train_runtime']:.0f} seconds used for training.")
    print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
    print(f"Peak reserved memory = {used_memory} GB.")
    print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
    print(f"Peak reserved memory % of max memory = {used_percentage} %.")
    print(f"Peak reserved memory for training % of max memory = {training_memory_percentage} %.")

## Evaluate the fine-tuned model

Load the saved model and run inference on a clinical task to see how it performs.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load fine-tuned model (replace with your HF model path)
fine_tuned_model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
ft_tokenizer = AutoTokenizer.from_pretrained(output_dir)

In [ ]:
def run_episode(env, model, tokenizer, task_id, max_turns=25):
    """Run a complete EHR episode with the fine-tuned model."""
    obs_text = env.reset(task_id=task_id)
    print(f"Task: {task_id}")
    print(f"Observation: {obs_text}\n")

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Complete the clinical task.\n\nEnvironment: {obs_text}"},
    ]

    for turn in range(max_turns):
        if env.done:
            break

        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=128)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        print(f"Turn {turn}: {generated_text.strip()[:120]}")

        action = parse_action(generated_text)
        if action is None:
            env_feedback = "ERROR: Could not parse action."
        else:
            action_type = action["type"]
            if action_type == "goto":
                env_feedback = env.navigate(action["url"])
            elif action_type == "click":
                env_feedback = env.click(action["selector"])
            elif action_type == "fill":
                env_feedback = env.type_text(action["selector"], action["text"])
            elif action_type == "keypress":
                env_feedback = env.press_key(action["key"])
            else:
                env_feedback = "ERROR: Unknown action."

        print(f"  → {env_feedback.split(chr(10))[0]}")

        messages.append({"role": "assistant", "content": generated_text})
        messages.append({"role": "user", "content": env_feedback})

    print(f"\nResult: reward={env.cumulative_reward:.2f}, steps={env.step_count}, done={env.done}")
    print(f"Rubric progress: {env.rubric_progress}")


# Run on a sample task
eval_env = EHRGymEnv(base_url=ehrgym_url)
try:
    run_episode(eval_env, fine_tuned_model, ft_tokenizer, "aki-chart-review")
finally:
    eval_env.close()

---

## What's next?

- **Scale up the model**: Try `Qwen/Qwen3.5-7B` or larger for better clinical reasoning
- **More training steps**: Increase `dataset_size` and `num_train_epochs`
- **Multi-GPU**: Use `vllm_mode="server"` with `trl vllm-serve` for distributed training
- **Local Docker**: Run EHRGym locally for faster episode throughput:
  ```bash
  docker run -d -p 7860:7860 registry.hf.space/openenv-community-ehrgym:latest
  ```
- **Custom tasks**: Add your own clinical scenarios in `tasks/examples/`

For more details, see the [EHRGym README](https://huggingface.co/spaces/openenv-community/EHRGym) and [TRL OpenEnv docs](https://huggingface.co/docs/trl/openenv).